# AI FinOps - Pruebas Automáticas de Modelos
Este notebook realiza las consultas automáticamente con cada uno de los 3 modelos a través del proxy FinOps y genera los datos de ejecución.

In [ ]:
import os
import json
import time
import httpx
import pandas as pd

# Definición de tarifas por millón de tokens para cada modelo
PRICING = {
    "llama3.2:3b": {"input": 0.06 / 1_000_000, "output": 0.06 / 1_000_000},
    "mistral:7b": {"input": 0.24 / 1_000_000, "output": 0.24 / 1_000_000},
    "llama-3.1-8b-instant": {"input": 0.05 / 1_000_000, "output": 0.08 / 1_000_000}
}

# Mapeo de departamento a usuario
DEPT_TO_USER = {
    "marketing": "ana",
    "desarrollo": "carlos",
    "RRHH": "paco"
}

def calculate_cost(model, prompt_tokens, completion_tokens):
    rates = PRICING.get(model, {"input": 0.0, "output": 0.0})
    return (prompt_tokens * rates["input"]) + (completion_tokens * rates["output"])

### Cargar Preguntas
Cargamos las preguntas de `preguntas.json` para procesar.

In [ ]:
# Cargar preguntas
with open("preguntas.json", "r", encoding="utf-8") as f:
    preguntas = json.load(f)

print(f"Total preguntas cargadas: {len(preguntas)}")
df_preguntas = pd.DataFrame(preguntas)
df_preguntas.head()

### Configurar Parámetros de Ejecución
Configura cuántas preguntas ejecutar por departamento y el límite de tokens (`max_tokens`) para agilizar el procesamiento (muy recomendado si los modelos corren localmente en CPU).

In [ ]:
# Ajusta esto a un número (ej: 2) si deseas probar rápido o None para ejecutar todas.
LIMIT_PER_DEPARTMENT = 2 

# Ajusta el límite de tokens de generación para agilizar la prueba. 
# Pon None para respuesta completa sin límite.
MAX_TOKENS = 15 

### Ejecutar Pruebas
Esta celda recorre las preguntas y realiza las peticiones a la API del proxy para cada modelo.

In [ ]:
# Filtrar preguntas por departamento si hay un límite configurado
if LIMIT_PER_DEPARTMENT is not None:
    preguntas_filtradas = []
    counts = {}
    for q in preguntas:
        dept = q.get("departamento", "unknown")
        counts[dept] = counts.get(dept, 0)
        if counts[dept] < LIMIT_PER_DEPARTMENT:
            preguntas_filtradas.append(q)
            counts[dept] += 1
    preguntas_a_ejecutar = preguntas_filtradas
else:
    preguntas_a_ejecutar = preguntas

models = ["llama3.2:3b", "mistral:7b", "llama-3.1-8b-instant"]
resultados = []

total_peticiones = len(preguntas_a_ejecutar) * len(models)
peticion_actual = 0

print(f"Iniciando consultas automáticas para {len(preguntas_a_ejecutar)} preguntas y {len(models)} modelos...")

for q in preguntas_a_ejecutar:
    dept = q.get("departamento", "unknown")
    query_text = q.get("query", "").strip()
    if not query_text:
        continue
        
    usuario = DEPT_TO_USER.get(dept, "default")
    
    for model in models:
        peticion_actual += 1
        print(f"[{peticion_actual}/{total_peticiones}] {model} | {dept} | {query_text[:50]}...")
        
        url = f"http://localhost:8000/api/v1/{model}/chat/completions"
        headers = {
            "Content-Type": "application/json",
            "x-username": usuario
        }
        payload = {
            "messages": [{"role": "user", "content": query_text}]
        }
        if MAX_TOKENS is not None:
            payload["max_tokens"] = MAX_TOKENS
            
        start_time = time.time()
        status_code = None
        prompt_tokens = 0
        completion_tokens = 0
        cost = 0.0
        response_text = ""
        
        try:
            # Petición HTTP
            response = httpx.post(url, headers=headers, json=payload, timeout=60.0)
            latency = time.time() - start_time
            status_code = response.status_code
            
            if response.status_code == 200:
                resp_json = response.json()
                choices = resp_json.get("choices", [])
                if choices:
                    response_text = choices[0].get("message", {}).get("content", "")
                
                usage = resp_json.get("usage", {})
                prompt_tokens = usage.get("prompt_tokens", 0)
                completion_tokens = usage.get("completion_tokens", 0)
                cost = calculate_cost(model, prompt_tokens, completion_tokens)
            else:
                response_text = response.text
                print(f"  Error {response.status_code}: {response_text[:150]}")
        except Exception as e:
            latency = time.time() - start_time
            response_text = str(e)
            print(f"  Excepción: {e}")
            
        resultados.append({
            "departamento": dept,
            "usuario": usuario,
            "query": query_text,
            "model": model,
            "status_code": status_code,
            "latency_s": latency,
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "cost": cost,
            "response_preview": response_text[:150]
        })
        time.sleep(0.3)

# Guardar los resultados
with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, indent=2, ensure_ascii=False)
    
print("\n¡Ejecución completada!")
print("Resultados guardados en resultados.json")

### Analizar Resultados
Cargamos los resultados generados en un DataFrame de pandas para analizar las métricas por modelo.

In [ ]:
df_res = pd.DataFrame(resultados)
if not df_res.empty:
    resumen = df_res.groupby("model").agg(
        total_peticiones=("query", "count"),
        latencia_promedio_s=("latency_s", "mean"),
        tokens_entrada_total=("prompt_tokens", "sum"),
        tokens_salida_total=("completion_tokens", "sum"),
        coste_total_usd=("cost", "sum")
    )
    display(resumen)
else:
    print("No hay resultados cargados.")